# Analyse de l'anémie infantile — EDS Cameroun 2018
## 0. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

from scipy import stats
from statsmodels.miscmodels.ordinal_model import OrderedModel
import statsmodels.formula.api as smf
import statsmodels.api as sm

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

## 1. Chargement des données

In [ ]:

# os.chdir("C:\\Users\\user\\Desktop\\saint jean\\semetre2\\fichierstati\\HR_IR_MR_PR")
# os.chdir("C:\\Users\\mngono\\Desktop\\tp-sn-python\\stat")
# os.chdir("C:\\Users\\user\\Desktop\\saint jean\\semetre2\\STAT\\statistiqueTp\\stat")
os.chdir("C:\\Users\\mngono\\Desktop\\modele-regression-ordinal-multiniveau")


dataIR = pd.read_stata("CMIR71FL.DTA", convert_categoricals=False)
 

In [ ]:
# ── Fichier KR (enfants) ─────────────────────────────
dataKR = pd.read_spss("CMKR71FL.SAV")


In [ ]:
# print("Shape dataHR:", dataHR.shape)
print("Shape dataIR:", dataIR.shape)
# print("Shape dataPR:", dataPR.shape)
print("Shape dataKR:", dataKR.shape)

In [ ]:
# print("Types dataHR :")
# print(dataHR.dtypes)
print("\nTypes dataIR :")
print(dataIR.dtypes)
# print("\nTypes dataPR :")
# print(dataPR.dtypes)
print("\nTypes dataKR :")
print(dataKR.dtypes)

In [ ]:
# Convertir toutes les colonnes KR en minuscules
dataKR.columns = dataKR.columns.str.lower()


# Vérification
print("KR colonnes (après correction) :")
print(dataKR.columns[:10].tolist())

print("KR :", dataKR.shape)

In [ ]:
# Garder uniquement les variables utiles
variables_KR = [
    'caseid', 'v001', 'v002', 'v005',      # identifiants
    # 'hw56', 
    'hw57',                          # anémie
    'b4', 'b19', 'hw70', 'hw72',            # enfant
    'h11', 'h22', 'h31', 'h42', 'h43', 'm4', # santé enfant
    'v012', 'v106', 'v133', 'v151',          # mère
    'v201', 'v457', 'v025', 'v024',          # mère & contexte
    'v190', 'v113', 'v116', 'v445'           # ménage
]

# Garder seulement les variables qui existent
variables_dispo = [v for v in variables_KR if v in dataKR.columns]
variables_absentes = [v for v in variables_KR if v not in dataKR.columns]

print(" Variables disponibles :", variables_dispo)
print(" Variables absentes    :", variables_absentes)

In [ ]:
rename_map = {
    # identifiants
    'caseid': 'id_menage',
    'v001': 'cluster',
    'v002': 'menage',
    'v005': 'poids_echantillon',

    # santé enfant (anémie / nutrition)
    # 'hw56': 'hemoglobine',
    'hw57': 'anemie',
    'hw70': 'zscore_taille_age',
    'hw72': 'zscore_poids_taille',

    # enfant
    'b4': 'sexe_enfant',
    'b19': 'age_enfant_mois',
    'm4': 'type_allaitement',

    # santé enfant (questions)
    'h11': 'diarrhee',
    
    'h22': 'fievre',
    'h31': 'toux',
    'h42': 'prise_fer',
    'h43': 'deparasitage',

    # mère
    'v012': 'age_mere',
    'v106': 'niveau_instruction',
    'v133': 'annees_education',
    # sexe_chef_menage
    'v151': 'sexe_chef_menage',

    # contexte enfant / ménage
    'v201': 'nombre_enfants_nes',
    'v457': 'anemie_mere',
    'v025': 'milieu_residence',
    'v024': 'region',

    # richesse / conditions ménage
    'v190': 'indice_richesse',
    'v113': 'source_eau',
    'v116': 'type_toilettes',
    'v445': 'BMI'
}


In [ ]:
# diarrhee
dataKR["h11"].value_counts()

In [ ]:
# fievre
dataKR["h22"].value_counts()

In [ ]:
# toux
dataKR["h31"].value_counts()

In [ ]:
# deparasitage
dataKR["h43"].value_counts()

In [ ]:
# prise_fer
dataKR["h42"].value_counts()

In [ ]:
dataKR.head()

In [ ]:
# ── Créer le dataframe ─────────────────────────────
dfKR = dataKR[variables_dispo].rename(columns=rename_map)

In [ ]:
dfKR.describe()

In [ ]:
# df['hw56'].unique()

In [ ]:


# ── Nettoyage des flags DHS sur dfKR (après renommage) ──
#  Bug 1 corrigé : liste explicite des colonnes numériques à nettoyer
cols_flags = ['zscore_taille_age', 'zscore_poids_taille', 'BMI']

for col in cols_flags:
    #  Bug 2 corrigé : vérifier dans dfKR (noms renommés)
    if col in dfKR.columns:
        # Bug 3 corrigé : nettoyer dfKR, pas dataKR
        #conversion numérique
        dfKR[col] = pd.to_numeric(dfKR[col], errors='coerce')
        #suppression codes DHS invalides en remplacant par nan
        dfKR[col] = dfKR[col].replace([9996, 9997, 9998, 9999], np.nan)
        n = dfKR[col].isna().sum()
        print(f"{col:<26} → {n:,} manquants ({n/len(dfKR)*100:.1f}%)")
        
        
        
# 2. Correction des échelles (UNE SEULE FOIS hors boucle)
if 'BMI' in dfKR.columns:
    dfKR['BMI'] = dfKR['BMI'] / 100

if 'zscore_taille_age' in dfKR.columns:
    dfKR['zscore_taille_age'] = dfKR['zscore_taille_age'] / 100

if 'zscore_poids_taille' in dfKR.columns:
    dfKR['zscore_poids_taille'] = dfKR['zscore_poids_taille'] / 100

if 'poids_echantillon' in dfKR.columns:
    dfKR['poids_echantillon'] = dfKR['poids_echantillon'] / 1000000

In [ ]:
dfKR.isnull().sum()

In [ ]:
dfKR.isnull().sum()  / len(dfKR)

In [ ]:
#LES VARIABLES ayant un des lignes nuls avec un pourcentages elevee

#toujours supprimer les nan de la variables y
df_model = dfKR.dropna(subset=['anemie'])

In [ ]:
df_model.isnull().sum()  / len(dfKR)

In [ ]:
df_model.dtypes

##imputation des variables avec lignes nan

In [ ]:


# Liste des variables pour verifier quel loi suivre chaque variable
variablesQantitatives = [
'zscore_taille_age','zscore_poids_taille','BMI'
]

variables_categorielles = [
    'anemie_mere'
]
    
# ── CONTINUES ──
for col in variablesQantitatives:

    if col in df_model.columns:

        data = pd.to_numeric(df_model[col], errors='coerce')

        plt.figure(figsize=(8,5))
        sns.histplot(data, kde=True)
        plt.title(f"Histogramme de {col}")
        plt.xlabel(col)
        plt.ylabel("Fréquence")
        plt.show()


# ── CATÉGORIELLES ──
for col in variables_categorielles:

    if col in df_model.columns:

        data = df_model[col]

        plt.figure(figsize=(8,5))
        sns.countplot(x=data)
        plt.title(f"Distribution de {col}")
        plt.xlabel(col)
        plt.ylabel("Fréquence")
        plt.xticks(rotation=45)
        plt.show()

##IMPUTATION

In [ ]:
# ── Z-SCORES (normaux) → moyenne ──
cols_cluster = ['zscore_taille_age', 'zscore_poids_taille']

for col in cols_cluster:

    df_model[col] = pd.to_numeric(df_model[col], errors='coerce')

    mean_cluster = df_model.groupby('cluster')[col].transform('mean')
    mean_global = df_model[col].mean()

    df_model[col] = df_model[col].fillna(mean_cluster)
    df_model[col] = df_model[col].fillna(mean_global)

    print(f"{col:<26} → NA restants : {df_model[col].isna().sum()}")


# ── BMI (asymétrique) → médiane ──
df_model['BMI'] = pd.to_numeric(df_model['BMI'], errors='coerce')

median_cluster = df_model.groupby('cluster')['BMI'].transform('median')
median_global = df_model['BMI'].median()

df_model['BMI'] = df_model['BMI'].fillna(median_cluster)
df_model['BMI'] = df_model['BMI'].fillna(median_global)

print(f"{'BMI':<26} → NA restants : {df_model['BMI'].isna().sum()}")


# ── anemie_mere (catégorielle) → mode ──
mode_cluster = df_model.groupby('cluster')['anemie_mere'].transform(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
)

mode_global = df_model['anemie_mere'].mode()[0]

df_model['anemie_mere'] = df_model['anemie_mere'].fillna(mode_cluster)
df_model['anemie_mere'] = df_model['anemie_mere'].fillna(mode_global)

print(
    f"{'anemie_mere':<26} → NA restants : "
    f"{df_model['anemie_mere'].isna().sum()}"
)

In [ ]:
(df_model.isnull().sum() / len(df_model)) * 100

In [ ]:
print(df_model['sexe_enfant'].value_counts(dropna=False))

In [ ]:
print(df_model['anemie'].value_counts())
print(df_model['anemie'].cat.categories)
print(df_model['anemie_mere'].value_counts())

In [ ]:
df_model["diarrhee"].value_counts()

In [ ]:
df_model["toux"].value_counts()

In [ ]:
df_model["fievre"].value_counts()

## 4.  mapping ou Encodage des variables catégorielles

In [ ]:
# ── Mappings des variables catégorielles ───────────
#permet de definir lordre de donne comme on veux faire un modele ordinal multiniveau
mappings = {
    'sexe_enfant': {'Male': 1, 'Female': 0},
    'milieu_residence' :{'Urban': 0, 'Rural': 1},
    # 'Plus pauvre': 1, 'Pauvre': 2, 'Moyen': 3, 'Riche': 4, 'Plus riche': 5
    'indice_richesse': {'Poorest': 1, 'Poorer': 2, 'Middle': 3, 'Richer': 4, 'Richest': 5},
    #   'Aucune instruction': 0, 'Primaire': 1,  'Secondaire': 2, 'Supérieur': 3
    'niveau_instruction': {'No education': 0, 'Primary': 1,'Secondary': 2,   'Higher': 3},
    # sexe_chef_menage
    'sexe_chef_menage': {'Male': 1, 'Female': 0},
    # 'Pas d’anémie': 0, 'Anémie légère': 1, 'Anémie modérée': 2,'Anémie sévère': 3
    'anemie':{'Not anemic': 0, 'Mild': 1,'Moderate': 2,'Severe': 3},
    'anemie_mere': {'Not anemic': 0, 'Mild': 1,'Moderate': 2,'Severe': 3 },
    # Région (10 régions Cameroun)
    'region': {
    'Far-North'                    : 1,
    'North'                        : 2,
    'Adamawa'                      : 3,
    'Centre (without Yaounde)'     : 4,
    'Yaounde'                      : 4,  
    'East'                         : 5,
    'West'                         : 6,
    'Littoral (without Douala)'    : 7,
    'Douala'                       : 7, 
    'North-West'                   : 8,
    'South-West'                   : 9,  
    'South'                        : 10,  
},
    # Consultation santé (h42 = prise de fer)
    'prise_fer': { 'No' : 0, 'Yes': 1, "Don't know": 8},
    # Hospitalisation (h43 = déparasitage)
    'deparasitage': {'No': 0,'Yes': 1, "Don't know": 8},
    # Type allaitement (pas accouchement !)
    'type_allaitement': { 'Never breastfed': 0,'Still breastfeeding': 1,
        'Ever breastfed, not currently breastfeeding': 2,
    },
'diarrhee' : {'No': 0, 'Yes, last two weeks': 2, "Don't know": 8},
'fievre'   : {'No': 0, 'Yes': 1, "Don't know": 8},
'toux'     : {'No': 0, 'Yes, last two weeks': 2, "Don't know": 8},
 }


# ── Application des mappings ──────────────────────

for col, mapping in mappings.items():

    if col in df_model.columns:
        
        # garder les vrais NaN + nettoyer espaces
        df_model[col] = df_model[col].astype(str).str.strip()

        # appliquer mapping
        df_model[col] = df_model[col].map(mapping)
        # conversion finale en numérique si possible
        #df_model[col] = pd.to_numeric(df_model[col], errors="ignore")
        df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

        n_na = df_model[col].isna().sum()

        print(f"{col:<26} → mappé | NA : {n_na:,}")

    else:
        print(f"{col:<26} → absent de df_model")





In [ ]:
df_model["diarrhee"].value_counts()

In [ ]:
# correction

cols_restants = ['prise_fer', 'deparasitage', 'diarrhee', 'fievre', 'toux']

for col in cols_restants:

    if col in dfKR.columns:

        # mode global SAFE (évite erreurs)
        if dfKR[col].dropna().empty:
            print(f"{col} →  colonne vide après nettoyage")
            continue

        mode_global = dfKR[col].mode().iloc[0]

        n_before = dfKR[col].isna().sum()

        dfKR[col] = dfKR[col].fillna(mode_global)

        n_after = dfKR[col].isna().sum()

        print(f"{col:<20} → mode global = {mode_global} | NA: {n_before} → {n_after}")
        

In [ ]:
print(f"Dataset final : {df_model.shape[0]} observations × {df_model.shape[1]} variables\n")
print(f"{'Variable':<25} {'Type'}")
print("-" * 40)
for col in df_model.columns:
    print(f"{col:<25} {df_model[col].dtype}")

In [ ]:
df_model.isna().sum()

## 5. Statistiques descriptives

In [ ]:

# 80 cas sévères sur 3 987 = 2% → trop faible pour un modèle ordinal stable
# Fusion : Modérée (2) + Sévère (3) → 2

df_model['anemie'] = df_model['anemie'].replace({3: 2})

print("Distribution après regroupement :")
counts = df_model['anemie'].value_counts().sort_index()
print(counts)

# ── Graphique ──
ax = counts.plot(
    kind='bar',
    color=['#2ecc71', '#f39c12', '#e74c3c'],
    figsize=(6,4)
)

ax.set_title("Anémie (après regroupement)")
ax.set_xlabel("Classe d'anémie")
ax.set_ylabel("Effectif")
ax.grid(axis='y', alpha=0.3)

plt.show()


In [ ]:

summary = pd.DataFrame({
    "Indicateur": [
        "Total observations",
        "Nombre de clusters",
        "Nombre de régions",
        "Moyenne enfants / cluster"
    ],
    "Valeur": [
        f"{len(df_model):,}",
        df_model['cluster'].nunique(),
        df_model['region'].nunique(),
        round(len(df_model) / df_model['cluster'].nunique(), 1)
    ]
})

print(summary)

In [ ]:
# Niveau 1 : Enfant
niveau_1 = [
    'sexe_enfant',
    'age_enfant_mois',
    'zscore_taille_age',
    'zscore_poids_taille',
    'diarrhee',
    'fievre',
    'toux',
    'prise_fer',
    'deparasitage',
    'type_allaitement'
]

# Niveau 2 : Ménage / Cluster
niveau_2 = [
    'age_mere',
    'niveau_instruction',
    'annees_education',
    'sexe_chef_menage',
    'nombre_enfants_nes',
    'anemie_mere',
    'indice_richesse',
    'source_eau',
    'type_toilettes'
]

# Effet aléatoire niveau 2
groupe_niveau_2 = 'cluster'

# Variable contextuelle
niveau_3 = [
    'milieu_residence'
]

# Effet aléatoire niveau 3
groupe_niveau_3 = 'region'

In [ ]:
print(df_model[['cluster', 'region']].head())
print("Nombre de régions :", df_model['region'].nunique())
print("Nombre de clusters :", df_model['cluster'].nunique())

In [ ]:
resume_region = (
    df_model
    .groupby('region')
    .agg(
        nb_enfants=('cluster', 'size'),
        nb_clusters=('cluster', 'nunique')
    )
    .reset_index()
)

resume_region['moy_enfants_par_cluster'] = (
    resume_region['nb_enfants'] / resume_region['nb_clusters']
)

print(resume_region)

In [ ]:
df_model.groupby('cluster').size().describe()

revenir

In [ ]:
# dfKR['sexe_enfant'].value_counts(dropna=False)
# print(dfKR['sexe_enfant'])
print(f"Échantillon final : {df_model.shape[0]} observations")

# ══════════════════════════════════════════════════════
# ANALYSE UNIVARIÉE
# ══════════════════════════════════════════════════════

In [ ]:
# ── Variables quantitatives ───────────────────────────
vars_quant = [
    'age_enfant_mois', 'age_mere', 'BMI',
    'zscore_taille_age', 'zscore_poids_taille','nombre_enfants_nes'
]

print("=" * 65)
print(" VARIABLES QUANTITATIVES")
print("=" * 65)
print(f"{'Variable':<25} {'Moy':>8} {'Méd':>8} {'ET':>8} {'Min':>8} {'Max':>8}")

for col in vars_quant:
    if col in df_model.columns:
        d = df_model[col]

        print(f"{col:<25} {d.mean():>8.2f} {d.median():>8.2f} "
              f"{d.std():>8.2f} {d.min():>8.2f} {d.max():>8.2f}")

In [ ]:
# ── Variables catégorielles ───────────────────────────
vars_cat = [
    'anemie', 'sexe_enfant', 'milieu_residence', 'indice_richesse',
    'niveau_instruction', 'anemie_mere','fievre', 'diarrhee', 
    'toux','prise_fer', 'deparasitage','type_allaitement', 'sexe_chef_menage'
]

labels_map = {
    'anemie'           : {0:'Aucune', 1:'Légère', 2:'Modérée/Sévère'},
    'sexe_enfant'      : {1:'Masculin', 0:'Féminin'},
    'milieu_residence' : {0:'Urbain', 1:'Rural'},
    'indice_richesse'  : {1:'Très pauvre', 2:'Pauvre', 3:'Moyen', 4:'Riche', 5:'Très riche'},
    'niveau_instruction': {0:'Aucun', 1:'Primaire', 2:'Secondaire', 3:'Supérieur'},
    'anemie_mere'      : {0:'Aucune', 1:'Légère', 2:'Modérée/Sévère'},

    # symptômes santé
    'fievre'           : {0:'Non', 1:'Oui'},
    'prise_fer'        : {0:'Non', 1:'Oui'},

    #  attention ici tes codes sont différents
    'diarrhee'         : {0:'Non', 2:'Oui'},
    'toux'             : {0:'Non', 2:'Oui'},

    'deparasitage'     : {0:'Non', 1:'Oui'},

    'type_allaitement' : {0:'Jamais', 1:'En cours', 2:'Sevré'},
    'sexe_chef_menage' : {1:'Masculin', 0:'Féminin'},
}

print("\n" + "=" * 65)
print(" VARIABLES CATÉGORIELLES")
print("=" * 65)

for col in vars_cat:
    if col in df_model.columns:
        print(f"\n{col}")

        vc = df_model[col].value_counts().sort_index()

        for val, cnt in vc.items():
            lab = labels_map.get(col, {}).get(val, str(val))
            pct = cnt / len(df_model) * 100
            bar = "█" * int(pct / 3)

            print(f"  {lab:<15} : {cnt:>6} ({pct:5.1f}%) {bar}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle("Analyse Univariée — Enfants < 5 ans (EDS Cameroun)",
             fontsize=14, fontweight='bold')

color_hist = '#3498db'
color_cat1 = '#2ecc71'
color_cat2 = '#f39c12'
color_cat4 = '#e74c3c'

# ─────────────────────────────────────────────
# 1 — Âge enfant
# ─────────────────────────────────────────────
axes[0,0].hist(df_model['age_enfant_mois'].dropna(),
               bins=20, color='#1abc9c', edgecolor='black')
axes[0,0].set_title("Âge de l'enfant")
axes[0,0].set_xlabel('Âge (mois)')

# ─────────────────────────────────────────────
# 2 — Sexe enfant
# ─────────────────────────────────────────────
sex = df_model['sexe_enfant'].map({1:'Garçon', 0:'Fille'}).value_counts()

axes[0,1].pie(sex.values,
              labels=sex.index,
              autopct='%1.1f%%',
              colors=[color_cat1, color_cat4],
              startangle=90)

axes[0,1].set_title("Sexe de l'enfant")

# ─────────────────────────────────────────────
# 3 — Anémie (variable cible)
# ─────────────────────────────────────────────
anemie = df_model['anemie'].value_counts().sort_index()

axes[0,2].bar(anemie.index,
              anemie.values,
              color=['#2ecc71','#f39c12','#e74c3c'],
              edgecolor='black')

axes[0,2].set_title("Anémie (variable cible)")
axes[0,2].set_xlabel("Classe")

# ─────────────────────────────────────────────
# 4 — Milieu de résidence
# ─────────────────────────────────────────────
milieu = df_model['milieu_residence'].map({0:'Urbain', 1:'Rural'}).value_counts()

axes[1,0].pie(milieu.values,
              labels=milieu.index,
              autopct='%1.1f%%',
              colors=[color_cat1, color_cat2],
              startangle=90)

axes[1,0].set_title('Milieu de résidence')

# ─────────────────────────────────────────────
# 5 — Indice de richesse
# ─────────────────────────────────────────────
richesse = df_model['indice_richesse'].value_counts().sort_index()

axes[1,1].bar(richesse.index,
              richesse.values,
              color=['#d73027','#fc8d59','#fee08b','#91cf60','#1a9850'],
              edgecolor='black')

axes[1,1].set_title('Indice de richesse')
axes[1,1].set_xlabel('1=Très pauvre → 5=Très riche')

# ─────────────────────────────────────────────
# 6 — Éducation de la mère
# ─────────────────────────────────────────────
edu = df_model['niveau_instruction'].value_counts().sort_index()

axes[1,2].bar(edu.index,
              edu.values,
              color=['#8e44ad','#3498db','#f39c12','#2ecc71'],
              edgecolor='black')

axes[1,2].set_title('Éducation de la mère')
axes[1,2].set_xlabel('Niveau')
axes[1,2].tick_params(axis='x', rotation=15)

# ─────────────────────────────────────────────

plt.tight_layout()
plt.savefig('univarie.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Analyse bivariée

In [ ]:
# # ══════════════════════════════════════════════════════
# #           ÉTAPE  — ANALYSE BIVARIÉE
# # ══════════════════════════════════════════════════════
from scipy.stats import chi2_contingency, kruskal
from scipy.stats import chi2_contingency

# sécurité
df_model = df_model[df_model['anemie'].notna()].copy()
df_model['anemie'] = df_model['anemie'].astype(int)

vars_cat = {
    'sexe_enfant': 'Sexe enfant',
    'milieu_residence': 'Milieu de résidence',
    'indice_richesse': 'Indice de richesse',
    'niveau_instruction': 'Éducation de la mère',
    'anemie_mere': 'Anémie de la mère',
    'diarrhee': 'Diarrhée',
    'fievre': 'Fièvre',
    'toux': 'Toux',
    'prise_fer': 'Prise de fer',
    'deparasitage': 'Déparasitage',
    
    # Variables manquantes
    'type_allaitement': 'Type d’allaitement',
    'sexe_chef_menage': 'Sexe du chef de ménage',
    'source_eau': 'Source d’eau',
    'type_toilettes': 'Type de toilettes',
    'region': 'Région'
    
}

print(f"\n{'Variable':<25} {'Chi²':>10} {'p-value':>12} {'Signif':>8}")
print("-" * 60)

for var, label in vars_cat.items():
    
    if var in df_model.columns:
        
        tab = pd.crosstab(df_model[var], df_model['anemie'])

        chi2, p, dof, exp = chi2_contingency(tab)

        sig = "✔" if p < 0.05 else "✘"

        print(f"{label:<25} {chi2:>10.2f} {p:>12.4f} {sig:>8}")

In [ ]:
# from scipy.stats import kruskal

# Hypothèses du test de Kruskal-Wallis
# Ce test vérifie si une variable quantitative a la même distribution selon plusieurs groupes.

# Hypothèse nulle (H0)
# Les distributions (ou médianes) de la variable sont identiques dans tous les groupes d’anémie.

# Hypothèse alternative (H1)
# Au moins un groupe présente une distribution différente.

# on rejette H0 différence significative entre groupes

# ── Kruskal-Wallis — Variables quantitatives vs anémie ────────

# sécurité
df_model = df_model[df_model['anemie'].notna()].copy()
df_model['anemie'] = df_model['anemie'].astype(int)

# ── Variables quantitatives ──
vars_quant = {
    'age_enfant_mois'     : "Âge de l'enfant",
    'age_mere'            : 'Âge de la mère',
    'BMI'                 : 'IMC de la mère',
    'zscore_taille_age'   : 'Z-score T/A',
    'zscore_poids_taille' : 'Z-score P/T',
    'nombre_enfants_nes'  : "Nombre d'enfants",
    
    # # add   
    # 'annees_education': "Années d'éducation"
}

print(f"\n{'Variable':<25} {'H (Kruskal)':>12} {'p-value':>10} {'Sig':>6}")
print("-" * 55)

# groupes réels présents dans les données
groupes_anemie = sorted(df_model['anemie'].unique())

for var, label in vars_quant.items():
    
    if var in df_model.columns:
        
        groupes = [
            df_model[df_model['anemie'] == g][var].dropna()
            for g in groupes_anemie
        ]
        
        stat, p = kruskal(*groupes)
        sig = "✔" if p < 0.05 else "✘"

        print(f"{label:<25} {stat:>12.2f} {p:>10.4f} {sig:>6}")

# ══════════════════════════════════════════════════════
#           ANALYSE DE CORRÉLATION — paire à paire et SPEARMAN
# ══════════════════════════════════════════════════════

In [ ]:
# Corrélation de Spearman

# Elle sert à mesurer :

# la relation monotone entre DEUX variables quantitatives.

# H0 il n’existe pas de corrélation entre les deux variables. ρ = 0
# H1 il existe une corrélation. ρ ≠ 0


from scipy.stats import spearmanr

# variables quantitatives (corrigées)
vars_num = [
    'age_enfant_mois',
    'BMI',
    'age_mere',
    'zscore_taille_age',
    'zscore_poids_taille',
    'nombre_enfants_nes'
]

print(f"\n{'Variable 1':<30} {'Variable 2':<30} {'Rho':>8} {'p-value':>10} {'Sig':>6}")
print("-"*95)

for i in range(len(vars_num)):
    for j in range(i+1, len(vars_num)):
        
        var1 = vars_num[i]
        var2 = vars_num[j]

        if var1 in df_model.columns and var2 in df_model.columns:
            
            temp = df_model[[var1, var2]].dropna()

            rho, p = spearmanr(temp[var1], temp[var2])

            sig = "✔" if p < 0.05 else "✘"

            print(f"{var1:<30} {var2:<30} {rho:>8.3f} {p:>10.4f} {sig:>6}")

##Matrice de corrélation

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

vars_num = [
    'age_enfant_mois',
    'BMI',
    'age_mere',
    'zscore_taille_age',
    'zscore_poids_taille',
    'nombre_enfants_nes'
]

# garder seulement colonnes existantes
vars_num = [v for v in vars_num if v in df_model.columns]

corr = df_model[vars_num].corr(method='spearman')

print(corr)

plt.figure(figsize=(10,7))

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5
)

plt.title("Corrélation de Spearman")
plt.show()

In [ ]:
## 7. Test de multicolinéarité (VIF)

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Variables explicatives
vars_explicatives = [
    'sexe_enfant',
    'milieu_residence',
    'indice_richesse',
    'niveau_instruction',
    'anemie_mere',
    'diarrhee',
    'fievre',
    'deparasitage',
    'type_allaitement',
    'source_eau',
    'type_toilettes',
    'region',
    'age_enfant_mois',
    'age_mere',
    'BMI',
    'zscore_taille_age',
    'zscore_poids_taille'
]

# 1. Copie dataset
X = df_model[vars_explicatives].copy()

# 2. One-hot encoding
X = pd.get_dummies(X, drop_first=True)

# 3. FORCE conversion float (IMPORTANT FIX)
X = X.apply(pd.to_numeric, errors='coerce')

# 4. Remplacer inf par NaN
X = X.replace([np.inf, -np.inf], np.nan)

# 5. Supprimer NA
X = X.dropna()

# 6. Supprimer colonnes constantes
X = X.loc[:, X.nunique() > 1]

# 7. FORCE conversion finale en float (CRUCIAL)
X = X.astype(float)

# 8. Reset index
X = X.reset_index(drop=True)

# 9. Calcul VIF SAFE
vif_df = pd.DataFrame()
vif_df["Variable"] = X.columns
vif_df["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

# 10. Tri
vif_df = vif_df.sort_values("VIF", ascending=False)

print(vif_df)

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Variables explicatives
vars_explicatives = [
    'sexe_enfant',
    'milieu_residence',
    'indice_richesse',
    'niveau_instruction',
    'anemie_mere',
    'diarrhee',
    'fievre',
    'deparasitage',
    'type_allaitement',
    # 'source_eau',
    # 'type_toilettes',
    'region',
    'age_enfant_mois',
    'age_mere',
    # 'BMI',
    'zscore_taille_age',
    'zscore_poids_taille'
]

# 1. Copie dataset
X = df_model[vars_explicatives].copy()

# 2. One-hot encoding
X = pd.get_dummies(X, drop_first=True)

# 3. FORCE conversion float (IMPORTANT FIX)
X = X.apply(pd.to_numeric, errors='coerce')

# 4. Remplacer inf par NaN
X = X.replace([np.inf, -np.inf], np.nan)

# 5. Supprimer NA
X = X.dropna()

# 6. Supprimer colonnes constantes
X = X.loc[:, X.nunique() > 1]

# 7. FORCE conversion finale en float (CRUCIAL)
X = X.astype(float)

# 8. Reset index
X = X.reset_index(drop=True)

# 9. Calcul VIF SAFE
vif_df = pd.DataFrame()
vif_df["Variable"] = X.columns
vif_df["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

# 10. Tri
vif_df = vif_df.sort_values("VIF", ascending=False)

print(vif_df)

## 8. Construction du dataset final

In [ ]:
# final_vars = [
#     'sexe_enfant',
#     'age_enfant_mois',
#     'zscore_taille_age',
#     'zscore_poids_taille',
#     'diarrhee',
#     'fievre',
#     'deparasitage',
#     'age_mere',
#     'anemie_mere',
#     'indice_richesse',
#     'niveau_instruction',
#     'milieu_residence',
#     'region',
#     'type_allaitement'
# ]

final_vars = [
    'sexe_enfant',
    'age_enfant_mois',
    'diarrhee',
    'fievre',
    'deparasitage',
    'age_mere',
    'anemie_mere',
    'indice_richesse',
    'niveau_instruction',
    'milieu_residence',
    'region',
    'type_allaitement'
]

In [ ]:
# CREER MON DATA SET FINAL
df_final_anemie = df_model[final_vars + ["anemie", "cluster"]].copy()

In [ ]:
df_final_anemie = pd.get_dummies(df_final_anemie, drop_first=True)
df_final_anemie = df_final_anemie.dropna()

In [ ]:
# FICHIER TELECHARGEABLE
df_final_anemie.to_csv("df_final_anemie.csv", index=False)

In [ ]:
# modele ordinal multiniveau a ete faites avec R
import requests
def predict_clmm(sample):
        url = "http://localhost:8001:predict_clmm"
        r = requests.post(url, data=sample)
        return r.json()

# ou ca 
# clmm_preds = pd.read_csv("clmm_preds.csv")


## 9. Machine Learning — Prédiction du statut anémique
### 9.1 Préparation Train/Test

In [ ]:
# ==============================================================
# MACHINE LEARNING — PRÉDICTION DU STATUT ANÉMIQUE
# Random Forest + XGBoost + Régression Logistique Multinomiale
# EDS Cameroun 2018
# ==============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report,
                             confusion_matrix,
                             accuracy_score, f1_score)


# ==============================================================
# ÉTAPE 1 — TRAIN / TEST SPLIT
# ==============================================================

# X = df_final_anemie.drop(columns=['anemie'])

X = df_final_anemie.drop(columns=['anemie', 'cluster'])
y = df_final_anemie['anemie'].astype(int)

# Encodage des variables catégorielles
X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    # permet de separer le train et le test
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

print(X_train.dtypes.unique())

# Nombre total d'observations
n_total = len(X)

# Nombre d'observations dans chaque ensemble
n_train = len(X_train)
n_test = len(X_test)

# Pourcentages
pct_train = (n_train / n_total) * 100
pct_test = (n_test / n_total) * 100

print(f"Total : {n_total}")
print(f"Train : {n_train} ({pct_train:.2f}%)")
print(f"Test  : {n_test} ({pct_test:.2f}%)")

In [ ]:
# calculer le pourcentage

print(X_train.isna().sum().sum())
print(X_test.isna().sum().sum())

## Modeles non optimises

In [ ]:
# randon forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, log_loss

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

# Probabilités (nécessaires pour ROC AUC et Log Loss)
y_proba_rf = rf.predict_proba(X_test)

# Évaluer la qualité des prédictions
# Ces métriques servent à mesurer les performances du modèle sur les données de test

print("Accuracy :", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

# ROC AUC
print("ROC AUC :", roc_auc_score(y_test, y_proba_rf, multi_class='ovr'))

# Log Loss
print("Log Loss :", log_loss(y_test, y_proba_rf))



# Accuracy : 0.4974937343358396
#               precision    recall  f1-score   support

#            0       0.52      0.69      0.59       339
#            1       0.46      0.16      0.23       204
#            2       0.47      0.52      0.49       255

#     accuracy                           0.50       798
#    macro avg       0.48      0.45      0.44       798
# weighted avg       0.49      0.50      0.47       798

# ROC AUC : 0.6218866118842279
# Log Loss : 1.038834556543038


In [ ]:
plt.figure(figsize=(6,4))

plt.hist(y_test, alpha=0.5, label="Réel", density=True)
plt.hist(y_pred_rf, alpha=0.5, label="Prédit", density=True)

plt.xlabel("Classe d'anémie")
plt.ylabel("Densité")
plt.title("Distribution Réel vs Prédit (Random Forest)")
plt.legend()
plt.show()

In [ ]:
# MATRICE DE CONFUSION
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_rf)
print(cm)

# [[233  21  85]
#  [109  32  63]
#  [106  17 132]]

In [ ]:
#  xgboost 
import sys
!{sys.executable} -m pip install xgboost imbalanced-learn

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    log_loss
)
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

# 1. Encodage des labels
le = LabelEncoder()
y_enc = le.fit_transform(y)

# 2. Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc,
    test_size=0.2,
    random_state=42,
    stratify=y_enc
)

# 3. SMOTE (UNIQUEMENT SUR TRAIN)
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# 4. Modèle XGBoost
xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    objective='multi:softprob',
    num_class=3,
    random_state=42,
    eval_metric='mlogloss'
)

# 5. Entraînement SUR DONNÉES BALANCÉES
xgb.fit(X_train_res, y_train_res)

# 6. Prédiction
y_pred = xgb.predict(X_test)

# Probabilités (IMPORTANT pour ROC-AUC et Log Loss)
y_proba = xgb.predict_proba(X_test)

# 7. Évaluation
print("Accuracy :", accuracy_score(y_test, y_pred))
print("F1 macro :", f1_score(y_test, y_pred, average='macro'))

print("\nClassification report :")
print(classification_report(y_test, y_pred))

print("\nConfusion matrix :")
print(confusion_matrix(y_test, y_pred))

# 8. ROC AUC
print("\nROC AUC :", roc_auc_score(y_test, y_proba, multi_class='ovr'))

# 9. Log Loss
print("Log Loss :", log_loss(y_test, y_proba))


# Classification report :
#               precision    recall  f1-score   support

#            0       0.52      0.52      0.52       339
#            1       0.31      0.27      0.29       204
#            2       0.45      0.49      0.47       255

#     accuracy                           0.45       798
#    macro avg       0.42      0.43      0.42       798
# weighted avg       0.44      0.45      0.44       798


# Confusion matrix :
# [[177  75  87]
#  [ 82  55  67]
#  [ 81  50 124]]

# ROC AUC : 0.5983673840030946
# Log Loss : 1.138192650516049

In [ ]:
# LogisticRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    roc_auc_score,
    log_loss
)

lr = LogisticRegression(
    multi_class='multinomial', solver='lbfgs',
    max_iter=1000, class_weight='balanced', random_state=42
)

lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Probabilités (IMPORTANT pour ROC-AUC et Log Loss)
y_proba_lr = lr.predict_proba(X_test)

print("── Régression Logistique Multinomiale ──")
print(f"Accuracy : {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"F1 Macro : {f1_score(y_test, y_pred_lr, average='macro'):.4f}")

print(classification_report(
    y_test, y_pred_lr,
    target_names=['Pas anémie', 'Légère', 'Modérée-Sévère']
))

# ROC AUC (multi-classe)
print("\nROC AUC :", roc_auc_score(y_test, y_proba_lr, multi_class='ovr'))

# Log Loss
print("Log Loss :", log_loss(y_test, y_proba_lr))



# ── Régression Logistique Multinomiale ──
# Accuracy : 0.4561
# F1 Macro : 0.4261
#                 precision    recall  f1-score   support

#     Pas anémie       0.55      0.54      0.54       339
#         Légère       0.28      0.22      0.25       204
# Modérée-Sévère       0.45      0.54      0.49       255

#       accuracy                           0.46       798
#      macro avg       0.43      0.43      0.43       798
#   weighted avg       0.45      0.46      0.45       798


# ROC AUC : 0.610258964798753
# Log Loss : 1.0584912498984906

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)

print("── Matrice de confusion (Logistic Regression) ──")
print(cm_lr)

# ── Matrice de confusion (Logistic Regression) ──
# [[182  73  84]
#  [ 76  45  83]
#  [ 75  43 137]]

## comparaison des modele

In [ ]:
results = {
    "Logistic Regression": {
        "accuracy": accuracy_score(y_test, y_pred_lr),
        "f1": f1_score(y_test, y_pred_lr, average='macro'),
        "roc_auc": roc_auc_score(y_test, y_proba_lr, multi_class='ovr'),
        "log_loss": log_loss(y_test, y_proba_lr)
    },
    "Random Forest": {
        "accuracy": accuracy_score(y_test, y_pred_rf),
        "f1": f1_score(y_test, y_pred_rf, average='macro'),
        "roc_auc": roc_auc_score(y_test, y_proba_rf, multi_class='ovr'),
        "log_loss": log_loss(y_test, y_proba_rf)
    },
    "XGBoost": {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred, average='macro'),
        "roc_auc": roc_auc_score(y_test, y_proba, multi_class='ovr'),
        "log_loss": log_loss(y_test, y_proba)
    }
}

df_results = pd.DataFrame(results).T
print(df_results)

df_results[['accuracy', 'f1', 'roc_auc']].plot(kind='bar', figsize=(10,6))
plt.title("Comparaison des modèles")
plt.ylabel("Score")
plt.xticks(rotation=45)
plt.show()

In [ ]:
#resultat dans un tableau
results = pd.DataFrame({
    "Modèle": ["Random Forest", "XGBoost", "Logistic Regression"],
    "Accuracy": [0.472, 0.457, 0.432],
    "F1-score": [0.420, 0.435, 0.401],
    "ROC-AUC": [0.614, 0.608, 0.605],
    "Log Loss": [1.062, 1.103, 1.062]
})

# afficher proprement
print(results)

# ==============================================================
# IMPORTANCE DES VARIABLES (Random Forest)
# ==============================================================

In [ ]:
# # Calculer l'importance des variables du Random Forest :
importance = pd.DataFrame({
    "Variable": X_train.columns,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

# print(importance)

fig, ax = plt.subplots(figsize=(10, 7))

importance.sort_values("Importance", ascending=True)\
    .tail(15)\
    .plot(kind='barh',
          x="Variable",
          y="Importance",
          ax=ax,
          color='steelblue',
          alpha=0.8)

ax.set_title(
    'Top 15 Variables importantes — Random Forest\nEDS Cameroun 2018',
    fontweight='bold'
)

ax.set_xlabel('Importance')
ax.set_ylabel('Variables')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('importance_variables.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Prédictions train
y_pred_train = rf.predict(X_train)

# Accuracy train
train_acc = accuracy_score(y_train, y_pred_train)

# Accuracy test
test_acc = accuracy_score(y_test, y_pred_rf)

# Écart
gap = train_acc - test_acc

print(f"Accuracy train : {train_acc:.4f}")
print(f"Accuracy test  : {test_acc:.4f}")
print(f"Gap            : {gap:.4f}")

In [ ]:
# Vérifier la distribution des classes
print(y.value_counts())
print(y.value_counts(normalize=True) * 100)

### 9.2 Random Forest optimisé

In [ ]:
# RANDOM FOREST OPTIMISÉ (ANTI-OVERFITTING)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

from imblearn.over_sampling import SMOTE

# 1. SMOTE (UNIQUEMENT TRAIN)
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# 2. Modèle
rf = RandomForestClassifier(
    # permet doptimiser
    # n_estimators=400,        # assez d'arbres
    # max_depth=10,            # limite profondeur
    # min_samples_split=20,    # évite splits trop spécifiques
    # min_samples_leaf=10,     # stabilise les feuilles
    # pour rendre ca performant
    # diminue
    n_estimators=400,        # assez d'arbres
    max_depth=4,            # limite profondeur
    # augmente
    min_samples_split=30,    # évite splits trop spécifiques
    min_samples_leaf=15,     # stabilise les feuilles
    
    
    max_features='sqrt',     # réduit corrélation des arbres
    class_weight='balanced', # important pour déséquilibre
    random_state=42,
    n_jobs=-1
)

# 3. Entraînement SUR DONNÉES BALANCÉES
rf.fit(X_train_res, y_train_res)

y_pred_rf = rf.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred_rf))
print("F1 macro :", f1_score(y_test, y_pred_rf, average='macro'))
print(classification_report(y_test, y_pred_rf))

# Accuracy : 0.47117794486215536
# F1 macro : 0.4350979107660096
#               precision    recall  f1-score   support

#            0       0.54      0.55      0.55       339
#            1       0.30      0.20      0.24       204
#            2       0.47      0.58      0.52       255

#     accuracy                           0.47       798
#    macro avg       0.44      0.44      0.44       798
# weighted avg       0.46      0.47      0.46       798


#APRES AVOIR MODIFIER LES DONNES DESMETRIC

# Accuracy : 0.4523809523809524
# F1 macro : 0.41106903478860163
#               precision    recall  f1-score   support

#            0       0.53      0.57      0.55       339
#            1       0.26      0.17      0.20       204
#            2       0.44      0.53      0.48       255

#     accuracy                           0.45       798
#    macro avg       0.41      0.42      0.41       798
# weighted avg       0.43      0.45      0.44       798

## Matrice de confusion de random forest optimise

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_rf)

print("Matrice de confusion :")
print(cm)

# Matrice de confusion :
# [[188  68  83]
#  [ 80  41  83]
#  [ 81  27 147]]

# Matrice de confusion :
# [[192  66  81]
#  [ 80  34  90]
#  [ 87  33 135]]

## apprentissage du modele de random forest optimise

In [ ]:
# Prédictions train
y_pred_train = rf.predict(X_train)

# Accuracy train
train_acc = accuracy_score(y_train, y_pred_train)

# Accuracy test
test_acc = accuracy_score(y_test, y_pred_rf)

# Écart
gap = train_acc - test_acc

print(f"Accuracy train : {train_acc:.4f}")
print(f"Accuracy test  : {test_acc:.4f}")
print(f"Gap            : {gap:.4f}")

# Accuracy train : 0.5644
# Accuracy test  : 0.4712
# Gap            : 0.0933

# Accuracy train : 0.5428
# Accuracy test  : 0.4524
# Gap            : 0.0904

In [ ]:
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)

In [ ]:
# ── CLMM R traduit en Python ──────────────────────────
CLMM_COEFS = {
    "sexe_enfant":        0.1152,
    "age_enfant_mois":   -0.0276,
    "diarrhee":           0.0236,
    "fievre":             0.2706,
    "deparasitage":      -0.1013,
    "age_mere":          -0.0057,
    "anemie_mere":        0.4052,
    "indice_richesse":   -0.1702,
    "niveau_instruction":-0.1287,
    "milieu_residence":   0.1270,
    "region":             0.0140,
    "type_allaitement":   0.0962,
}
THRESHOLDS = {"0|1": -1.4410, "1|2": -0.2179}

def predict_clmm(data: PatientData) -> dict:
    eta = sum(
        CLMM_COEFS[col] * getattr(data, col)
        for col in CLMM_COEFS
    )
    t1, t2 = THRESHOLDS["0|1"], THRESHOLDS["1|2"]
    p0 = 1 / (1 + np.exp(-(t1 - eta)))
    p1 = 1 / (1 + np.exp(-(t2 - eta))) - p0
    p2 = 1 - p0 - p1
    proba = [p0, p1, p2]
    pred  = int(np.argmax(proba))
    return {
        "prediction": pred,
        "label": LABELS[pred],
        "probabilites": {
            "pas_anemie":    round(p0, 4),
            "leger":         round(p1, 4),
            "modere_severe": round(p2, 4),
        }
    }

##MATRICE DE CONFUSION

In [ ]:
from sklearn.metrics import accuracy_score

# RF
rf_acc = accuracy_score(y_test, rf.predict(X_test))

# CLMM
clmm_preds = [
    predict_clmm(X_test.iloc[i].to_dict())["prediction"]
    for i in range(len(X_test))
]

clmm_acc = accuracy_score(y_test, clmm_preds)

print("RF Accuracy:", rf_acc)
print("CLMM Accuracy:", clmm_acc)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm_rf = confusion_matrix(y_test, rf.predict(X_test))

sns.heatmap(cm_rf, annot=True, fmt="d")
plt.title("Random Forest")
plt.show()

## 10. Synthèse comparative des modèles

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

acc_rf   = accuracy_score(y_test, y_pred_rf)
f1_rf    = f1_score(y_test, y_pred_rf, average='macro')

acc_lr   = accuracy_score(y_test, y_pred_lr)
f1_lr    = f1_score(y_test, y_pred_lr, average='macro')

# XGBoost : reconvertir labels encodés en classes originales
y_pred_xgb_orig = le.inverse_transform(y_pred)
acc_xgb  = accuracy_score(y_test, y_pred_xgb_orig)
f1_xgb   = f1_score(y_test, y_pred_xgb_orig, average='macro')

# Métriques CLMM R
loglik_r = -4031.778
k_params = 18      # 2 seuils + 16 coefficients
n_obs    = 3987
aic_r    = -2 * loglik_r + 2 * k_params
bic_r    = -2 * loglik_r + k_params * np.log(n_obs)

comparaison = pd.DataFrame({
    "Modèle": [
        "CLMM (R — ordinal multiniveau)",
        "Random Forest (Python)",
        "XGBoost (Python)",
        "Logistique Multinomiale (Python)"
    ],
    "Type": [
        "Statistique / Explicatif",
        "Machine Learning",
        "Machine Learning",
        "Statistique / Prédictif"
    ],
    "Effets_aléatoires": ["Oui (cluster)", "Non", "Non", "Non"],
    "Accuracy": ["N/A", f"{acc_rf:.4f}", f"{acc_xgb:.4f}", f"{acc_lr:.4f}"],
    "F1_macro":  ["N/A", f"{f1_rf:.4f}",  f"{f1_xgb:.4f}",  f"{f1_lr:.4f}"],
    "AIC / LogLik": [f"AIC={aic_r:.1f} / logLik={loglik_r}", "N/A", "N/A", "N/A"],
    "Interprétabilité": ["Très haute", "Faible", "Moyenne", "Haute"],
    "FastAPI": ["Oui (JSON coefs)", "Oui (.pkl)", "Oui (.pkl)", "Oui (.pkl)"]
})

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print(comparaison.to_string(index=False))